# 02_preprocess.py
## Split + preprocessing (tanpa data leakage).

### PENTING: 
Notebook lama mengimputasi median pada SELURUH dataset sebelum split,
ini menyebabkan data leakage (median test bocor ke train). Di sini kita lakukan
urutan yang benar:
  1. Split dulu (80/20, stratify)
  2. Imputer + scaler HANYA difit pada X_train, lalu transform X_test

Cara pakai:
  python scripts/02_preprocess.py


In [18]:
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [19]:
BASE_DIR = os.path.dirname(os.path.abspath(''))
DATA_DIR = os.path.join(BASE_DIR, "data")
OUT_DIR = os.path.join(BASE_DIR, "output")
MODEL_DIR = os.path.join(BASE_DIR, "model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [20]:
print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR: {OUT_DIR}")
print(f"MODEL_DIR: {MODEL_DIR}")

BASE_DIR: /home/krayza/Documents/Kuliah/Semester 5/Machine Learning/PJBL/projectML2
DATA_DIR: /home/krayza/Documents/Kuliah/Semester 5/Machine Learning/PJBL/projectML2/data
OUT_DIR: /home/krayza/Documents/Kuliah/Semester 5/Machine Learning/PJBL/projectML2/output
MODEL_DIR: /home/krayza/Documents/Kuliah/Semester 5/Machine Learning/PJBL/projectML2/model


In [21]:
TARGET_COL = "Potability"
TEST_SIZE = 0.2
RANDOM_STATE = 42

#### Part 1 -- Load Data

In [22]:
df = pd.read_csv(os.path.join(DATA_DIR, "water_potability.csv"))
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]
feature_cols = list(X.columns)


#### Part 2 -- Split Data

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} baris | Test: {X_test.shape[0]} baris")

Train: 2620 baris | Test: 656 baris


#### Part 3 -- imputer (median) + scaler

In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), feature_cols),
    ],
    remainder="drop",
)

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

# 03_train_evaluate.py
## Training 5 model + Threshold Tuning + Permutation Importance.

## Tujuan:
  1. Melatih 5 algoritma ML pada data train yang bersih (tanpa leakage).
  2. Membandingkan metrik evaluasi pada data test (Accuracy, Precision, Recall, F1, ROC-AUC).
  3. Menguji THRESHOLD TUNING (mengubah threshold dari 0.5 ke 0.6, 0.7, dll)
     khusus untuk mengurangi False Positive (FP) — air berbahaya dianggap layak.
  4. Menghitung Permutation Importance (pengaruh riil tiap fitur pada test set).

Cara pakai:
  python scripts/03_train_evaluate.py

In [26]:
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


#### Part 1 : Definisi 5 Model yang dibandingkan

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
models = {
    "SVM (RBF)": CalibratedClassifierCV(SVC(C=1.0, kernel="rbf", random_state=RANDOM_STATE), cv=5, method="sigmoid", ensemble=False),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE),
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

#### Part 2 : Cross Validation + Penentuan Model Terbaik ROC-AUC Based

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
cv_rows = []
oof_proba = {}   # prediksi out-of-fold tiap model

for name, model in models.items():
    fold_auc = []
    proba = np.zeros(len(y_train))

    for tr_idx, va_idx in cv.split(X_train, y_train):
        # preprocessing ikut dilatih ulang di dalam tiap fold, jadi tidak ada leakage
        pipe = make_pipeline(clone(preprocessor), clone(model))
        pipe.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])

        p = pipe.predict_proba(X_train.iloc[va_idx])[:, 1]
        proba[va_idx] = p
        fold_auc.append(roc_auc_score(y_train.iloc[va_idx], p))

    oof_proba[name] = proba

    # Fit pada full X_train untuk mengecek gap Train vs CV (overfitting check)
    full_pipe = make_pipeline(clone(preprocessor), clone(model))
    full_pipe.fit(X_train, y_train)
    p_train = full_pipe.predict_proba(X_train)[:, 1]

    cv_rows.append({
        "Model": name,
        "Train ROC-AUC": round(roc_auc_score(y_train, p_train), 4),
        "CV ROC-AUC (rata-rata)": round(np.mean(fold_auc), 4),
        "CV ROC-AUC (std)": round(np.std(fold_auc), 4),
    })

cv_df = pd.DataFrame(cv_rows).sort_values("CV ROC-AUC (rata-rata)", ascending=False)
cv_df

/home/krayza/Documents/Kuliah/Semester 5/Machine Learning/PJBL/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/krayza/Documents/Kuliah/Semester 5/Machine Learning/PJBL/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/krayza/Documents/Kuliah/Semester 5/Machine Learning/PJBL/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/kray

,Model,CV ROC-AUC (rata-rata),CV ROC-AUC (std)
0,SVM (RBF),0.7019,0.0179
1,Random Forest,0.6724,0.0129
3,Gradient Boosting,0.6408,0.0227
4,KNN,0.6299,0.0205
2,Logistic Regression,0.4774,0.0177


In [ ]:
best_model_name = cv_df.iloc[0]["Model"]
y_proba_oof = oof_proba[best_model_name]

# Sama dengan kode lamamu, hanya sumbernya diganti:
#   y_proba_best -> y_proba_oof   dan   y_test -> y_train
thresh_results = []
for th in [0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]:
    y_pred_th = (y_proba_oof >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, y_pred_th).ravel()
    thresh_results.append({
        "Threshold": th,
        "Accuracy": round(accuracy_score(y_train, y_pred_th), 4),
        "Precision": round(precision_score(y_train, y_pred_th, zero_division=0), 4),
        "Recall": round(recall_score(y_train, y_pred_th, zero_division=0), 4),
        "FP": fp, "FN": fn, "TP": tp, "TN": tn,
    })
thresh_df = pd.DataFrame(thresh_results)

# aturan pemilihan sama seperti sebelumnya (target FP turun >= 40%)
base_fp = thresh_df.loc[thresh_df["Threshold"] == 0.5, "FP"].values[0]
matching = thresh_df[thresh_df["FP"] <= base_fp * 0.6]
optimal_row = matching.iloc[0] if len(matching) > 0 else thresh_df.loc[thresh_df["FP"].idxmin()]
opt_th = float(optimal_row["Threshold"])
thresh_df

,Threshold,Accuracy,Precision,Recall,FP,FN,TP,TN
0,0.30,0.5779,0.4752,0.7877,889,217,805,709
1,0.40,0.6687,0.5781,0.5577,416,452,570,1182
2,0.50,0.6809,0.6598,0.3757,198,638,384,1400
3,0.55,0.6794,0.7078,0.3033,128,712,310,1470
4,0.60,0.6744,0.7493,0.2485,85,768,254,1513
5,0.65,0.6615,0.7647,0.1908,60,827,195,1538
6,0.70,0.6550,0.8073,0.1517,37,867,155,1561
7,0.75,0.6401,0.8211,0.0988,22,921,101,1576
8,0.80,0.6305,0.8068,0.0695,17,951,71,1581


In [ ]:
final_model = make_pipeline(clone(preprocessor), clone(models[best_model_name]))
final_model.fit(X_train, y_train)

proba_test = final_model.predict_proba(X_test)[:, 1]
y_pred_test = (proba_test >= opt_th).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
print("Threshold:", opt_th)
print("ROC-AUC  :", round(roc_auc_score(y_test, proba_test), 4))
print("Accuracy :", round(accuracy_score(y_test, y_pred_test), 4))
print("Precision:", round(precision_score(y_test, y_pred_test, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_pred_test), 4))
print("Specificity:", round(recall_score(y_test, y_pred_test, zero_division=0, pos_label=0), 4))
print("FP:", fp, "| FN:", fn, "| TP:", tp, "| TN:", tn)

# Simpan pipeline lengkap & metadata untuk aplikasi
joblib.dump(final_model, os.path.join(MODEL_DIR, "model_pipeline.joblib"))

with open(os.path.join(OUT_DIR, "feature_stats.json")) as f:
    feature_stats = json.load(f)

model_info = {
    "best_model_name": best_model_name,
    "default_threshold": 0.5,
    "recommended_threshold": opt_th,
    "final_test_metrics_at_recommended_th": {
        "Threshold": opt_th,
        "ROC_AUC": round(roc_auc_score(y_test, proba_test), 4),
        "Accuracy": round(accuracy_score(y_test, y_pred_test), 4),
        "Precision": round(precision_score(y_test, y_pred_test, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred_test, zero_division=0), 4),
        "Specificity": round(recall_score(y_test, y_pred_test, zero_division=0, pos_label=0), 4),
        "FP": int(fp), "FN": int(fn), "TP": int(tp), "TN": int(tn),
    },
    "feature_cols": feature_cols,
    "feature_stats": feature_stats,
}
with open(os.path.join(MODEL_DIR, "model_info.json"), "w") as f:
    json.dump(model_info, f, indent=2)

print("Saved: model/model_pipeline.joblib & model/model_info.json")

/home/krayza/Documents/Kuliah/Semester 5/Machine Learning/PJBL/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Threshold: 0.6
ROC-AUC  : 0.6487
Accuracy : 0.6662
Precision: 0.7033
Recall   : 0.25
Specificity: 0.9325
FP: 27 | FN: 192 | TP: 64 | TN: 373


#### test buat nyari param terbaik

In [29]:
from sklearn.model_selection import GridSearchCV

pipe = make_pipeline(clone(preprocessor), RandomForestClassifier(random_state=RANDOM_STATE))

param_grid = {
    "randomforestclassifier__n_estimators": [100, 300],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5],
}

search = GridSearchCV(pipe, param_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
search.fit(X_train, y_train)

print(search.best_params_)
print(search.best_score_)   # AUC CV dari kombinasi terbaik

{'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_leaf': 5, 'randomforestclassifier__n_estimators': 300}
0.686562259427626
